# Sistem Pencocokan Inquiry Barang Teknik dengan Katalog Produk

Notebook ini mengimplementasikan pipeline hybrid (rule-based + technical dictionary + fuzzy matching + machine learning) untuk mencocokkan inquiry customer dengan katalog produk perusahaan, sesuai proposal.

**Alur pipeline:**

1. Item Segmentation
2. Text Preprocessing
3. Technical Term Normalization
4. Unit and Standard Normalization
5. Attribute Extraction
6. Candidate Product Retrieval
7. Technical Attribute Matching
8. Ranking and Scoring
9. Product Recommendation

Dataset : 
- **`Catalog.csv`** — daftar produk resmi perusahaan, berisi kolom nama produk & SKU.
- **`Inquiry.csv`** — daftar inquiry/permintaan customer dalam bentuk teks bebas (1 kolom saja, tanpa struktur baku).

# TAHAP 1 - Load Data & Item Segmentation

In [1]:
import re
import pandas as pd
import numpy as np
from collections import defaultdict
from rapidfuzz import fuzz
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from urllib.parse import quote_plus

pd.set_option("display.max_colwidth", None)

# =========================================================
# TOGGLE SUMBER DATA: CSV (default, buat development/testing)
# atau DATABASE (PostgreSQL, buat production/data asli)
# =========================================================
USE_DATABASE = True

if USE_DATABASE:
    from sqlalchemy import create_engine

    DB_USER = "postgres"
    DB_PASSWORD = quote_plus("F@mily59")
    DB_HOST = "localhost"
    DB_PORT = "5432"
    DB_NAME = "inquiry_matching"

    engine = create_engine(
        f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
    )

    # Ambil catalog dari PostgreSQL
    catalog_df = pd.read_sql("SELECT product_name AS catalog_name, product_id AS sku FROM catalog", engine)

    # Inquiry masih dari CSV
    def load_single_column_text(path, col_name):
        with open(path, encoding="utf-8-sig") as f:
            lines = [line.strip() for line in f]
        lines = [line for line in lines if line != ""]
        header, rows = lines[0], lines[1:]
        return pd.DataFrame({col_name: rows})

    inquiry_df = load_single_column_text(
        "Inquary.csv",
        "inquiry_name"
    )

print("Catalog:", catalog_df.shape, "Inquiry:", inquiry_df.shape)

def segment_items(df, text_col, id_prefix, sku_col=None):
    items = []
    for idx, row in df.iterrows():
        text = str(row[text_col]).strip()
        if text == "" or text.lower() == "nan":
            continue
        item = {"id": f"{id_prefix}_{idx+1:05d}", "raw_text": text}
        if sku_col is not None:
            item["sku"] = str(row[sku_col]).strip()
        items.append(item)
    return items

inquiry_items = segment_items(inquiry_df, "inquiry_name", "INQ")
catalog_items = segment_items(catalog_df, "catalog_name", "CAT", sku_col="sku")
print("segmented:", len(inquiry_items), len(catalog_items))

d:\File Dimas\Projek Magang\baru\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Catalog: (1044, 2) Inquiry: (2339, 1)
segmented: 2339 1044


---
## Tahap 2 — Text Preprocessing

**Tujuan:** Menyamakan format teks mentah (`raw_text`) sebelum masuk ke tahap normalisasi istilah, supaya pencarian pola dengan regex di tahap-tahap berikutnya lebih akurat dan konsisten.

**Detail proses (`preprocess_text`), dijalankan berurutan:**
1. **Uppercase seluruh teks** — supaya pencocokan tidak sensitif huruf besar/kecil (`bv` sama dengan `BV`).
2. **Pisahkan angka yang menempel dengan satuan** — mis. `40BAR` → `40 BAR`, `PN16` → `PN 16`. Tanpa langkah ini, regex ekstraksi angka di tahap-tahap selanjutnya bisa gagal mengenali angkanya.
3. **Hilangkan titik yang bukan bagian dari angka desimal** — mis. `NO.5` → `NO 5`, tapi `4.5 INCH` tetap `4.5 INCH` (titik di antara dua digit dipertahankan karena itu titik desimal).
4. **Ganti tanda baca pemisah** (`, / ; : _ -`) dengan spasi — supaya `SS-316` dan `SS 316` diperlakukan sama.
5. **Buang semua karakter selain huruf, angka, titik, dan spasi** — membersihkan simbol aneh/sisa encoding.
6. **Rapikan spasi ganda** menjadi satu spasi, lalu `strip()`.

Daftar `KNOWN_UNITS` (KW, HP, RPM, PSI, BAR, dst.) digunakan khusus untuk langkah pemisahan angka-satuan di atas.

In [2]:
KNOWN_UNITS = ["KW", "HP", "RPM", "PSI", "BAR", "PN", "INCH", "MM", "CM", "KG", "LB", "V", "A", "HZ", "LPM"]
UNITS_PATTERN = "|".join(KNOWN_UNITS)

def preprocess_text(text: str) -> str:
    text = text.upper()
    text = re.sub(rf"(\d)({UNITS_PATTERN})\b", r"\1 \2", text)
    text = re.sub(rf"\b({UNITS_PATTERN})(\d)", r"\1 \2", text)
    text = re.sub(r"(?<!\d)\.(?!\d)", " ", text)
    text = re.sub(r"[,/;:_\-]+", " ", text)
    text = re.sub(r"[^A-Z0-9. ]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

for it in inquiry_items:
    it["preprocessed_text"] = preprocess_text(it["raw_text"])
for it in catalog_items:
    it["preprocessed_text"] = preprocess_text(it["raw_text"])

---
## Tahap 3 — Technical Term Normalization



**Tujuan:** Menerjemahkan berbagai singkatan, istilah lapangan, atau ejaan non-baku menjadi satu istilah, supaya inquiry dan katalog bisa dibandingkan dengan kosakata yang sama meskipun penulisannya berbeda-beda.

**Detail proses:**
- `SYNONYM_GROUPS` adalah kamus manual yang memetakan satu istilah baku (key) ke daftar sinonim/singkatannya (value). Contoh: `"BALL VALVE": ["BALL VLV", "BV", "B VALVE", "B V/V"]`, semua varian ini akan diseragamkan menjadi `"BALL VALVE"`.
- Dari `SYNONYM_GROUPS`, dibuat `TECHNICAL_DICTIONARY`, yaitu mapping terbalik dari **setiap sinonim individual → istilah baku**, supaya lookup per-istilah jadi O(1).
- `sorted_terms` mengurutkan seluruh istilah dari **paling panjang ke paling pendek**. Ini krusial: kalau istilah pendek (mis. `"BB"`) diganti duluan, dia bisa "memakan" sebagian dari istilah yang lebih panjang dan menyebabkan salah normalisasi. Dengan mengurutkan dari terpanjang, istilah yang lebih spesifik selalu diprioritaskan.
- `normalize_terms()` melakukan pencarian & penggantian dengan *word boundary* (`\b...\b`) agar tidak salah mengganti potongan kata yang kebetulan mengandung singkatan tersebut.

In [3]:
# =========================================================
# SYNONYM_GROUPS -- sama seperti sumber data katalog/inquiry,
# bisa dibaca dari CSV/hardcoded (default) atau dari database
# =========================================================
if USE_DATABASE:
    # struktur tabel: technical_synonyms(canonical_term, synonym)
    # satu canonical_term bisa punya banyak baris synonym
    synonym_df = pd.read_sql(
        "SELECT canonical_term, synonym FROM technical_synonyms", engine
    )
    TECHNICAL_DICTIONARY = dict(zip(synonym_df["synonym"], synonym_df["canonical_term"]))

else:
    SYNONYM_GROUPS = {
        "BEARING": ["BRG", "LAHER", "LAKER", "LAGER"],
        "BALL BEARING": ["BALL BRG","BB","BALL BRNG"],
        "DEEP GROOVE BALL BEARING": ["DGBB", "DEEP GROOVE", "DG BALL BEARING"],

        "VALVE": ["VLV", "V/V", "KATUP", "KLEP", "KERANGAN"],
        "BALL VALVE": ["BALL VLV", "BV", "B VALVE", "B V/V"],
        "GATE VALVE": ["GATE VLV", "GV", "SLUICE VALVE"],
        "GLOBE VALVE": ["GLOBE VLV", "GLV"],
        "CHECK VALVE": ["CHECK VLV", "CV", "NRV"],
        "BUTTERFLY VALVE": ["BFLY VLV", "BUTTERFLY VLV", "BVF", "BF VALVE"],

        "MOTOR": ["MTR", "MOT"],
        "PUMP": ["PMP", "POMPA", "PUMP UNIT", "PUMPING UNIT"],
        "CENTRIFUGAL PUMP": ["CENT PMP", "CENT PUMP", "CF PUMP", "CENTRIFUGAL PMP" ],
        "SUBMERSIBLE PUMP": ["SUB PMP", "SUB PUMP", "SUBMERSIBLE PMP", "SUB POMPA"],

        "CARBON STEEL": ["CS", "CARBON STL", "CARBON", "C STEEL"],
        "STAINLESS STEEL": ["SS", "STAINLESS", "S/STEEL", "STL SS"],
        "STAINLESS STEEL 316": ["SS316", "SS-316", "316SS" ],
        "STAINLESS STEEL 304": ["SS304", "SS-304", "304SS"],

        "THREE PHASE": ["3PH", "3PHS", "3 PH", "3-PH", "THREE PH" ],
        "SINGLE PHASE": ["1PH", "1PHS", "1 PH", "1-PH", "ONE PH" ],
        "4 POLE": ["4P","4PL", "4 PH", "4-PH", "FOUR PH"],
        "2 POLE": ["2P","2PL", "2 PH", "2-PH", "TWO PH"],
        "6 POLE": ["6P","6PL", "6 PH", "6-PH", "SIX PH"],
        "IMB3": ["B3", "IM B3", "IM-B3"],
        "IMB5": ["B5", "IM B5", "IM-B5"],
        "2Z": ["ZZ"],
        "MANUFACTURER": ["MFR"],

        "ELBOW": ["ELB", "L BOW", "EL", "EL B"],
        "REDUCER": ["RED", " REDUCING"],
        "CONCENTRIC": ["CONC", "CON", "CNCTC"],
        "BUTT WELD": ["BW", "B-W", "BUTTWELD"],
        "WELD NECK": ["WN"],
        "SLIP ON": ["SO", "SLIP-ON"],
        "BLIND": ["BL", "BLD"],
        "RAISED FACE": ["RF", "R/F"],
        "DEGREE": ["DEG", "DEG.", "DGR"],
        "LONG RADIUS": ["LR", "LONG RAD", "LRADIUS"],
    }

    TECHNICAL_DICTIONARY = {
        term: canonical
        for canonical, terms in SYNONYM_GROUPS.items()
        for term in terms
    }

sorted_terms = sorted(TECHNICAL_DICTIONARY.keys(), key=len, reverse=True)

def normalize_terms(text: str) -> str:
    for term in sorted_terms:
        canonical = TECHNICAL_DICTIONARY[term]
        if canonical.startswith(term + " "):
            next_word = canonical[len(term) + 1:].split()[0]
            pattern = r"\b" + re.escape(term) + r"\b(?!\s+" + re.escape(next_word) + r"\b)"
        else:
            pattern = r"\b" + re.escape(term) + r"\b"
        text = re.sub(pattern, canonical, text)
    return re.sub(r"\s+", " ", text).strip()

for it in inquiry_items:
    it["normalized_text"] = normalize_terms(it["preprocessed_text"])
for it in catalog_items:
    it["normalized_text"] = normalize_terms(it["preprocessed_text"])

print(f"Total sinonim aktif: {len(TECHNICAL_DICTIONARY)} (sumber: {'database' if USE_DATABASE else 'hardcoded dictionary'})")


Total sinonim aktif: 117 (sumber: database)


---
## Tahap 4 — Unit and Standard Normalization


In [4]:
NPS_TO_DN = {
    "1/8": "DN6", "1/4": "DN8", "3/8": "DN10", "1/2": "DN15",
    "3/4": "DN20", "1": "DN25", "1 1/4": "DN32", "1 1/2": "DN40",
    "2": "DN50", "2 1/2": "DN65", "3": "DN80", "4": "DN100",
}

def normalize_pressure(text: str) -> str:
    def psi_to_bar(m):
        return f"{round(float(m.group(1)) * 0.0689476, 2)} BAR"
    return re.sub(r"(\d+(?:\.\d+)?)\s*PSI", psi_to_bar, text)

def normalize_units(text: str) -> str:
    text = normalize_pressure(text)
    return re.sub(r"\s+", " ", text).strip()

for it in inquiry_items:
    it["unit_normalized_text"] = normalize_units(it["normalized_text"])
for it in catalog_items:
    it["unit_normalized_text"] = normalize_units(it["normalized_text"])

---
## Tahap 5 — Attribute Extraction


In [5]:
KNOWN_BRANDS = ["SKF", "NTN", "NSK", "FAG", "TIMKEN", "KOYO", "NACHI",
                "KITZ", "TOMOE", "ABB", "WEG", "EBARA", "TSURUMI", "ROTAN"]

def find_brand(text):
    return next((b for b in KNOWN_BRANDS if re.search(rf"\b{b}\b", text)), None)

def detect_category(text: str) -> str:
    if "FLANGE" in text: return "FLANGE"
    if "VALVE" in text: return "VALVE"
    if "MOTOR" in text: return "MOTOR"
    if "BEARING" in text: return "BEARING"
    if "PUMP" in text: return "PUMP"
    if re.search(r"\b(ELBOW|TEE|REDUCER)\b", text): return "FITTING"
    if "PIPE" in text: return "PIPE"
    return "UNKNOWN"

def extract_bearing_attrs(text):
    attrs = {}
    m = re.search(r"\b(\d{4})\b", text); attrs["model"] = m.group(1) if m else None
    m = re.search(r"\b(2Z|2RS1|ZZ|OPEN)\b", text); attrs["closure"] = m.group(1) if m else None
    m = re.search(r"\b(C[0-9])\b", text); attrs["clearance"] = m.group(1) if m else None
    attrs["brand"] = find_brand(text)
    return attrs

def extract_valve_attrs(text):
    attrs = {}
    m = re.search(r"\b(BALL|GATE|GLOBE|CHECK|BUTTERFLY)\s+VALVE\b", text)
    attrs["valve_type"] = f"{m.group(1)} VALVE" if m else None
    m = re.search(r"STAINLESS STEEL\s*(\d{3})", text)
    attrs["material"] = f"STAINLESS STEEL {m.group(1)}" if m else None

    m = re.search(r"\b(DN\d+)\b", text)
    if m:
        attrs["nominal_size"] = m.group(1)
    else:
        attrs["nominal_size"] = None
        for nps in sorted(NPS_TO_DN.keys(), key=len, reverse=True):
            if re.search(r"\b" + re.escape(nps) + r"\s*INCH\b", text):
                attrs["nominal_size"] = NPS_TO_DN[nps]
                break

    m = re.search(r"(\d+(?:\.\d+)?)\s*BAR", text); attrs["pressure_bar"] = float(m.group(1)) if m else None
    m = re.search(r"\bPN\s*(\d+)\b", text); attrs["pn_rating"] = f"PN{m.group(1)}" if m else None
    attrs["brand"] = find_brand(text)
    return attrs

def extract_motor_attrs(text):
    attrs = {}
    m = re.search(r"\b(THREE PHASE|SINGLE PHASE)\b", text); attrs["phase"] = m.group(1) if m else None
    m = re.search(r"(\d+(?:\.\d+)?)\s*KW", text); attrs["power_kw"] = float(m.group(1)) if m else None
    m = re.search(r"(\d+)\s*POLE", text); attrs["pole"] = int(m.group(1)) if m else None
    m = re.search(r"\b(IMB\d+)\b", text); attrs["mounting"] = m.group(1) if m else None
    attrs["brand"] = find_brand(text)
    return attrs

def extract_pump_attrs(text):
    attrs = {}
    m = re.search(r"\b(CENTRIFUGAL|SUBMERSIBLE|GEAR)\s+PUMP\b", text)
    attrs["pump_type"] = f"{m.group(1)} PUMP" if m else None
    m = re.search(r"\b(\d+)\s*X\s*(\d+)\b", text)
    attrs["port_size"] = f"{m.group(1)}X{m.group(2)}" if m else None
    m = re.search(r"\b(\d+(?:\.\d+)?)\s*INCH\b", text)
    attrs["size_inch"] = float(m.group(1)) if m else None
    m = re.search(r"(\d+)\s*RPM", text); attrs["rpm"] = int(m.group(1)) if m else None
    m = re.search(r"(\d+(?:\.\d+)?)\s*KW", text); attrs["power_kw"] = float(m.group(1)) if m else None
    m = re.search(r"(\d+)\s*LPM", text); attrs["capacity_lpm"] = int(m.group(1)) if m else None
    attrs["brand"] = find_brand(text)
    return attrs

def extract_pipe_attrs(text):
    attrs = {}
    m = re.search(r"(CARBON STEEL|STAINLESS STEEL\s*\d{3})", text); attrs["material"] = m.group(1) if m else None
    m = re.search(r"\b(\d+(?:\.\d+)?)\s*INCH\b", text); attrs["size_inch"] = float(m.group(1)) if m else None
    m = re.search(r"\bSCH\s*(\d+)\b", text); attrs["schedule"] = f"SCH{m.group(1)}" if m else None
    return attrs

def extract_fitting_attrs(text):
    attrs = {}
    m = re.search(r"\b(ELBOW|TEE|REDUCER)\b", text); attrs["fitting_type"] = m.group(1) if m else None
    m = re.search(r"(\d+)\s*DEGREE", text); attrs["angle"] = f"{m.group(1)} DEGREE" if m else None
    m = re.search(r"(CARBON STEEL|STAINLESS STEEL\s*\d{3})", text); attrs["material"] = m.group(1) if m else None
    m = re.search(r"\b(\d+(?:\.\d+)?)\s*X\s*(\d+(?:\.\d+)?)\s*INCH\b", text)
    if m:
        attrs["size_inch"] = f"{m.group(1)}X{m.group(2)}"
    else:
        m = re.search(r"\b(\d+(?:\.\d+)?)\s*INCH\b", text)
        attrs["size_inch"] = m.group(1) if m else None
    m = re.search(r"\bSCH\s*(\d+)\b", text); attrs["schedule"] = f"SCH{m.group(1)}" if m else None
    return attrs

def extract_flange_attrs(text):
    attrs = {}
    m = re.search(r"\b(WELD NECK|SLIP ON|BLIND)\s+FLANGE\b", text)
    attrs["flange_type"] = f"{m.group(1)} FLANGE" if m else None
    m = re.search(r"\b(RAISED FACE|FLAT FACE)\b", text); attrs["face_type"] = m.group(1) if m else None
    m = re.search(r"\b(\d+(?:\.\d+)?)\s*INCH\b", text); attrs["size_inch"] = float(m.group(1)) if m else None
    m = re.search(r"\bCLASS\s*(\d+)\b", text); attrs["class_rating"] = f"CLASS{m.group(1)}" if m else None
    return attrs

CATEGORY_CONFIG = {
    "BEARING": {"extractor": extract_bearing_attrs, "hard": ["model"]},
    "VALVE":   {"extractor": extract_valve_attrs,   "hard": ["valve_type", "nominal_size", "pressure_bar"]},
    "MOTOR":   {"extractor": extract_motor_attrs,   "hard": ["phase", "power_kw", "pole", "mounting"]},
    "PUMP":    {"extractor": extract_pump_attrs,    "hard": ["pump_type"]},
    "PIPE":    {"extractor": extract_pipe_attrs,    "hard": ["size_inch", "schedule"]},
    "FITTING": {"extractor": extract_fitting_attrs, "hard": ["fitting_type", "size_inch"]},
    "FLANGE":  {"extractor": extract_flange_attrs,  "hard": ["flange_type", "size_inch", "class_rating"]},
}

def extract_attributes(category, text):
    return CATEGORY_CONFIG[category]["extractor"](text) if category in CATEGORY_CONFIG else {}

for it in inquiry_items:
    it["category"] = detect_category(it["unit_normalized_text"])
    it["attributes"] = extract_attributes(it["category"], it["unit_normalized_text"])
for it in catalog_items:
    it["category"] = detect_category(it["unit_normalized_text"])
    it["attributes"] = extract_attributes(it["category"], it["unit_normalized_text"])

print(pd.Series([c["category"] for c in inquiry_items]).value_counts())

MOTOR      576
BEARING    336
FITTING    336
VALVE      315
PIPE       288
PUMP       180
UNKNOWN    157
FLANGE     151
Name: count, dtype: int64


---
## Tahap 6 — Candidate Product Retrieval


In [6]:
catalog_texts = [c["unit_normalized_text"] for c in catalog_items]
catalog_ids = [c["id"] for c in catalog_items]

vectorizer = TfidfVectorizer(analyzer="word", token_pattern=r"[A-Z0-9.]+")
catalog_tfidf_matrix = vectorizer.fit_transform(catalog_texts)

semantic_model = SentenceTransformer("all-MiniLM-L6-v2")
catalog_embeddings = semantic_model.encode(catalog_texts, show_progress_bar=True, normalize_embeddings=True)

catalog_by_category = defaultdict(list)
for idx, c in enumerate(catalog_items):
    catalog_by_category[c["category"]].append(idx)


def get_candidates(inquiry_item, top_n=20, fuzzy_weight=0.25, tfidf_weight=0.40, semantic_weight=0.35):
    text = inquiry_item["unit_normalized_text"]
    category = inquiry_item["category"]

    candidate_idx = catalog_by_category.get(category, [])
    if category == "UNKNOWN" or len(candidate_idx) == 0:
        candidate_idx = list(range(len(catalog_items)))

    tfidf_sims = cosine_similarity(vectorizer.transform([text]), catalog_tfidf_matrix[candidate_idx]).flatten()

    query_embedding = semantic_model.encode([text], normalize_embeddings=True)
    semantic_sims = cosine_similarity(query_embedding, catalog_embeddings[candidate_idx]).flatten()

    scored = []
    for local_i, global_i in enumerate(candidate_idx):
        tfidf_score = tfidf_sims[local_i]
        semantic_score = semantic_sims[local_i]
        fuzzy_score = fuzz.token_sort_ratio(text, catalog_texts[global_i]) / 100.0
        combined = (tfidf_weight * tfidf_score) + (fuzzy_weight * fuzzy_score) + (semantic_weight * semantic_score)
        scored.append((global_i, combined, tfidf_score, fuzzy_score, semantic_score))

    scored.sort(key=lambda x: x[1], reverse=True)
    top = scored[:top_n]

    return [
        {
            "catalog_id": catalog_ids[gi], "sku": catalog_items[gi].get("sku", ""),
            "catalog_text": catalog_items[gi]["unit_normalized_text"],
            "catalog_attributes": catalog_items[gi]["attributes"],
            "combined_score": round(combined, 4),
            "tfidf_score": round(tfidf_score, 4),
            "fuzzy_score": round(fuzzy_score, 4),
            "semantic_score": round(semantic_score, 4),
        }
        for gi, combined, tfidf_score, fuzzy_score, semantic_score in top
    ]


for it in inquiry_items:
    it["candidates"] = get_candidates(it)

Batches: 100%|██████████| 33/33 [00:04<00:00,  7.21it/s]


---
## Tahap 7 — Technical Attribute Matching


In [7]:
def match_attributes(category, inquiry_attrs, candidate_attrs):
    config = CATEGORY_CONFIG.get(category)
    if config is None:
        return {"is_valid": False}

    hard_keys = set(config["hard"])
    result = {}
    valid = True

    for key, inq_val in inquiry_attrs.items():
        cand_val = candidate_attrs.get(key)
        if key == "pressure_bar":
            status = "MEMENUHI" if (inq_val is not None and cand_val is not None and cand_val >= inq_val) else \
                     ("TIDAK MEMENUHI" if inq_val is not None and cand_val is not None else "TIDAK DIKETAHUI")
        else:
            status = "TIDAK DIKETAHUI" if inq_val is None else ("MATCH" if inq_val == cand_val else "TIDAK MATCH")
        result[key] = status
        if key in hard_keys and status in ("TIDAK MATCH", "TIDAK MEMENUHI"):
            valid = False

    result["is_valid"] = valid
    return result

for it in inquiry_items:
    for cand in it["candidates"]:
        cand["attribute_match"] = match_attributes(it["category"], it["attributes"], cand["catalog_attributes"])

---
## Tahap 8 — Ranking and Scoring

In [8]:
CATEGORY_ATTR_WEIGHTS = defaultdict(lambda: {"attribute_weight": 0.7, "retrieval_weight": 0.3})
CATEGORY_ATTR_WEIGHTS["UNKNOWN"] = {"attribute_weight": 0.0, "retrieval_weight": 1.0}
HARD_CONSTRAINT_PENALTY = 0.5

def compute_attribute_score(attribute_match):
    scored_keys = [k for k, v in attribute_match.items() if k != "is_valid" and v != "TIDAK DIKETAHUI"]
    if not scored_keys:
        return 0.0
    match_count = sum(1 for k in scored_keys if attribute_match[k] in ("MATCH", "MEMENUHI"))
    return match_count / len(scored_keys)

def rank_candidates(inquiry_item):
    weights = CATEGORY_ATTR_WEIGHTS[inquiry_item["category"]]
    for cand in inquiry_item["candidates"]:
        attr_score = compute_attribute_score(cand["attribute_match"])
        final_score = weights["attribute_weight"] * attr_score + weights["retrieval_weight"] * cand["combined_score"]
        if not cand["attribute_match"].get("is_valid", True):
            final_score *= HARD_CONSTRAINT_PENALTY
        cand["attribute_score"] = round(attr_score, 4)
        cand["final_score"] = round(final_score, 4)
    return sorted(inquiry_item["candidates"], key=lambda c: c["final_score"], reverse=True)

for it in inquiry_items:
    it["ranked_candidates"] = rank_candidates(it)

---
## Tahap 9 — Product Recommendation


In [9]:
def generate_reason(attribute_match):
    scored_keys = [k for k, v in attribute_match.items() if k != "is_valid" and v != "TIDAK DIKETAHUI"]
    if not scored_keys:
        return "Tidak ada atribut teknis yang dapat dibandingkan."
    matched = [k for k in scored_keys if attribute_match[k] in ("MATCH", "MEMENUHI")]
    unmatched = [k for k in scored_keys if attribute_match[k] not in ("MATCH", "MEMENUHI")]
    parts = []
    if matched: parts.append(f"{', '.join(matched)} sesuai dengan inquiry")
    if unmatched: parts.append(f"{', '.join(unmatched)} tidak sesuai")
    reason = "; ".join(parts) + "."
    if not attribute_match.get("is_valid", True):
        reason += " Catatan: melanggar hard constraint, sehingga skor diturunkan."
    return reason

def build_recommendation(inquiry_item, top_k=3):
    ranked = inquiry_item["ranked_candidates"][:top_k]
    recommendations = [{
        "rank": i + 1,
        "label": "Top 1 (Best Match)" if i == 0 else f"Top {i+1} (Alternative)",
        "sku": cand["sku"], "recommended_product": cand["catalog_text"],
        "match_score_pct": round(cand["final_score"] * 100, 1),
        "reason": generate_reason(cand["attribute_match"]),
    } for i, cand in enumerate(ranked)]
    return {"inquiry_id": inquiry_item["id"], "inquiry_input": inquiry_item["raw_text"],
            "category": inquiry_item["category"], "recommendations": recommendations}

for it in inquiry_items:
    it["final_recommendation"] = build_recommendation(it)

summary_rows = [{
    "Inquiry": it["final_recommendation"]["inquiry_input"],
    "SKU": it["final_recommendation"]["recommendations"][0]["sku"] if it["final_recommendation"]["recommendations"] else None,
    "Recommended Product": it["final_recommendation"]["recommendations"][0]["recommended_product"] if it["final_recommendation"]["recommendations"] else None,
    "Match Score": f"{it['final_recommendation']['recommendations'][0]['match_score_pct']}%" if it["final_recommendation"]["recommendations"] else None,
} for it in inquiry_items]

result_df = pd.DataFrame(summary_rows)
display(result_df.head(10))

,Inquiry,SKU,Recommended Product,Match Score
0,BEARING 6204 2Z C3 SKF DEEP GROOVE BALL,TVXX01XX0145,BALL VALVE V 3 AKU3TZM STAINLESS STEEL 304 DIA 2 KITZ,8.9%
1,DEEP GROOVE BALL BRG 6204 ZZ C3,TVXX01XX0051,BALL VALVE V 3 PE 2 STAINLESS STEEL 304,9.1%
2,SKF-DGBB 6204 ZZ C3,TVXX01XX0145,BALL VALVE V 3 AKU3TZM STAINLESS STEEL 304 DIA 2 KITZ,9.1%
3,ITEM DGBB 6204 2Z C3 MFR SKF,TVXX09XX2188,BALL VALVE 2 PIECES BODY CF3M BODY CF3M BALL SEAT MAT RPTFE ANSI150 2 HAND LEVER FLANGE COVNA,9.2%
4,SKF DGBB 6204 ZZ C3,TVXX01XX0145,BALL VALVE V 3 AKU3TZM STAINLESS STEEL 304 DIA 2 KITZ,9.1%
5,DGBB 6204 ZZ C3,TVXX01XX0051,BALL VALVE V 3 PE 2 STAINLESS STEEL 304,9.1%
6,ITEM DGBB 6204 ZZ C3 MFR SKF,TVXX09XX2188,BALL VALVE 2 PIECES BODY CF3M BODY CF3M BALL SEAT MAT RPTFE ANSI150 2 HAND LEVER FLANGE COVNA,9.2%
7,DEEP / GROOVE / BALL BEARING 6204 2Z C3,TVXX01XX0051,BALL VALVE V 3 PE 2 STAINLESS STEEL 304,9.1%
8,"DGBB, 6204 ZZ C3 MFR SKF",TVXX01XX0145,BALL VALVE V 3 AKU3TZM STAINLESS STEEL 304 DIA 2 KITZ,9.0%
9,DGBB 6204 2Z C3,TVXX01XX0051,BALL VALVE V 3 PE 2 STAINLESS STEEL 304,9.1%


# INPUT INQUIRY

In [10]:
def process_single_inquiry(raw_text, top_k=3):
    item = {"id": "MANUAL_INPUT", "raw_text": raw_text.strip()}
    item["preprocessed_text"] = preprocess_text(item["raw_text"])
    item["normalized_text"] = normalize_terms(item["preprocessed_text"])
    item["unit_normalized_text"] = normalize_units(item["normalized_text"])
    item["category"] = detect_category(item["unit_normalized_text"])
    item["attributes"] = extract_attributes(item["category"], item["unit_normalized_text"])
    item["candidates"] = get_candidates(item)
    for cand in item["candidates"]:
        cand["attribute_match"] = match_attributes(item["category"], item["attributes"], cand["catalog_attributes"])
    item["ranked_candidates"] = rank_candidates(item)
    item["final_recommendation"] = build_recommendation(item, top_k=top_k)
    return item

In [12]:
inquiry_name = 'PTY-30 STRAINER PN16-DN20, FLANGE, CAST IRON BODY, AISI 304 STRAINER & FILTER, AYVAS'
show_recomendation = 5

result = process_single_inquiry(inquiry_name, top_k=show_recomendation)

print("INQUIRY  :", result["raw_text"])
print("KATEGORI :", result["category"])
print()

table_rows = [{
    "Rank": rec["label"],
    "Recommended Product": rec["recommended_product"],
    "SKU": rec["sku"] if rec["sku"] else "(SKU tidak ditemukan)",
    "Match Score": f"{rec['match_score_pct']}%",
    "Alasan": rec["reason"],
} for rec in result["final_recommendation"]["recommendations"]]

recommendation_df = pd.DataFrame(table_rows)
styled = (recommendation_df.style
          .set_properties(**{'text-align': 'left'})
          .set_table_styles([{'selector': 'th', 'props': [('text-align', 'left')]}]))

display(styled)

INQUIRY  : PTY-30 STRAINER PN16-DN20, FLANGE, CAST IRON BODY, AISI 304 STRAINER & FILTER, AYVAS
KATEGORI : FLANGE



,Rank,Recommended Product,SKU,Match Score,Alasan
0,Top 1 (Best Match),PTY 30 STRAINER PN 16 DN20 FLANGE CAST IRON BODY AISI 304 STRAINER FILTER AYVAS,TFXX09XX0002,30.0%,Tidak ada atribut teknis yang dapat dibandingkan.
1,Top 2 (Alternative),PTY 30 STRAINER PN 16 DN40 FLANGE CAST IRON BODY AISI 304 STRAINER FILTER AYVAS,TVXX09XX0074,27.0%,Tidak ada atribut teknis yang dapat dibandingkan.
2,Top 3 (Alternative),PTY 30 STRAINER PN 16 DN25 FLANGE CAST IRON BODY AISI 304 STRAINER FILTER AYVAS,TVXX09XX0073,26.9%,Tidak ada atribut teknis yang dapat dibandingkan.
3,Top 4 (Alternative),PTY 30 STRAINER PN 16 DN50 FLANGE CAST IRON BODY AISI 304 STRAINER FILTER AYVAS,TFXX09XX0004,26.8%,Tidak ada atribut teknis yang dapat dibandingkan.
4,Top 5 (Alternative),PTY 30 STRAINER PN 16 DN80 FLANGE CAST IRON BODY AISI 304 STRAINER FILTER AYVAS,TFXX09XX0006,26.3%,Tidak ada atribut teknis yang dapat dibandingkan.
